<a href="https://colab.research.google.com/github/almendraapolaya/DI_Bootcamp_a/blob/main/Week_8/Day_2/Exercises%20/Exercises_XP_VDB_w8_d2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [1]:
%pip uninstall -y pydantic-core pydantic
%pip install -U "pydantic<2"
%pip install -U "faiss-cpu>=1.8.0" "chromadb==0.3.21"
%pip install -U "numpy<2" sentence-transformers transformers

  Using cached pydantic-1.10.26-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (155 kB)
Using cached pydantic-1.10.26-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (2.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.13 requires pydantic<3.0.0,>=2.0.0, but you have pydantic 1.10.26 which is incompatible.
pydantic-settings 2.14.0 requires pydantic>=2.7.0, but you have pydantic 1.10.26 which is incompatible.
langchain-core 1.3.1 requires pydantic<3.0.0,>=2.7.4, but you have pydantic 1.10.26 which is incompatible.
langsmith 0.7.34 requires pydantic<3,>=2, but you have pydantic 1.10.26 which is incompatible.
google-genai 1.68.0 requires pydantic<3.0.0,>=2.9.0, but you have pydantic 1.10.26 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 1.10.26 which is incompatible.
spacy 3

In [2]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display
os.makedirs('cache', exist_ok=True)


## 🌟 Exercise 1 · Data loading and preparation

In [4]:
data_path = 'labelled_newscatcher_dataset.csv'
pdf = pd.read_csv(data_path, sep=';')

if 'id' not in pdf.columns:
    pdf['id'] = range(len(pdf))

pdf_subset = pdf.head(1000).copy()

display(pdf_subset[['id', 'title']].head())


,id,title
0,0,A closer look at water-splitting's solar fuel ...
1,1,"An irresistible scent makes locusts swarm, stu..."
2,2,Artificial intelligence warning: AI will know ...
3,3,Glaciers Could Have Sculpted Mars Valleys: Study
4,4,Perseid meteor shower 2020: What time and how ...


## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [5]:
def example_create_fn(idx: int, text: str) -> InputExample:
    return InputExample(guid=str(idx), texts=[text], label=0.0)

faiss_train_examples = [
    example_create_fn(row['id'], row['title'])
    for _, row in pdf_subset.iterrows()
]

faiss_train_examples[:2]


In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')

titles_list = pdf_subset['title'].tolist()

faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)

print(f"Number of vectors: {len(faiss_title_embedding)}")
print(f"Dimensions per vector: {len(faiss_title_embedding[0])}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Number of vectors: 1000
Dimensions per vector: 384


## 🌟 Exercise 3 · FAISS indexing and search

In [7]:
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)
content_encoded_normalized = faiss_title_embedding.astype('float32')
faiss.normalize_L2(content_encoded_normalized)
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)
index_content.ntotal


1000

In [8]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):

    query_vector = model.encode([query]).astype('float32')

    faiss.normalize_L2(query_vector)

    sims, ids = index_content.search(query_vector, k)

    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()

    results['similarities'] = sims[0]

    return results.sort_values(by='similarities', ascending=False)

display(search_content('animal', pdf_to_index, k=5))


,topic,link,domain,published_date,title,lang,id,similarities
99,TECHNOLOGY,https://www.gematsu.com/2020/08/ghostwire-toky...,gematsu.com,2020-08-07 16:43:13,Ghostwire: Tokyo confirms dog petting,en,99,0.391902
176,TECHNOLOGY,https://www.pushsquare.com/news/2020/08/random...,pushsquare.com,2020-08-03 16:30:00,Random: You Can Pick Up and Pet Cats in Assass...,en,176,0.376784
762,SCIENCE,https://af.reuters.com/article/worldNews/idAFK...,af.reuters.com,2020-08-13 16:51:00,'Secret' life of sharks: Study reveals their s...,en,762,0.344059
928,SCIENCE,https://www.thecut.com/2020/08/scientists-say-...,thecut.com,2020-08-04 12:52:00,Just Let This Lizard Be a Dinosaur,en,928,0.317387
975,HEALTH,https://www.news-medical.net/news/20200813/Res...,news-medical.net,2020-08-13 05:18:00,Researchers explore social behavior of animals...,en,975,0.295497


## 🌟 Exercise 4 · ChromaDB collection and querying

In [9]:
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = 'my_news'
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)

collection.add(
    embeddings=faiss_title_embedding.tolist(),
    documents=pdf_subset['title'].tolist(),
    ids=[str(i) for i in pdf_subset['id']]
)

results = collection.query(
    query_texts=["animal"],
    n_results=5
)

print(json.dumps(results, indent=2))


ERROR:chromadb.telemetry.posthog:Failed to send telemetry event client_start: capture() takes 1 positional argument but 3 were given


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ERROR:chromadb.telemetry.posthog:Failed to send telemetry event collection_add: capture() takes 1 positional argument but 3 were given


{
  "ids": [
    [
      "176",
      "975",
      "99",
      "928",
      "762"
    ]
  ],
  "embeddings": null,
  "documents": [
    [
      "Random: You Can Pick Up and Pet Cats in Assassin's Creed Valhalla",
      "Researchers explore social behavior of animals toward emerging infectious diseases",
      "Ghostwire: Tokyo confirms dog petting",
      "Just Let This Lizard Be a Dinosaur",
      "'Secret' life of sharks: Study reveals their surprising social networks"
    ]
  ],
  "metadatas": [
    [
      null,
      null,
      null,
      null,
      null
    ]
  ],
  "distances": [
    [
      1.216195821762085,
      1.2464324235916138,
      1.3118828535079956,
      1.3652254343032837,
      1.409005880355835
    ]
  ]
}


## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [11]:
from transformers import pipeline

model_id = 'google/flan-t5-small'

pipe = pipeline("text-generation", model=model_id)

question = "What's the latest news on space development?"

context_docs = results['documents'][0][:3]
context = ' '.join(context_docs)

prompt = f"Answer the question using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:\n"

response = pipe(prompt, max_length=50)[0]['generated_text']

print(f"Prompt sent to model:\n{prompt}")
print("-" * 30)
print(f"Model Response: {response}")


model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausalLM', 'FalconForCausalLM',

Prompt sent to model:
Answer the question using only the context.
Context: Random: You Can Pick Up and Pet Cats in Assassin's Creed Valhalla Researchers explore social behavior of animals toward emerging infectious diseases Ghostwire: Tokyo confirms dog petting
Question: What's the latest news on space development?
Answer:

------------------------------
Model Response: Answer the question using only the context.
Context: Random: You Can Pick Up and Pet Cats in Assassin's Creed Valhalla Researchers explore social behavior of animals toward emerging infectious diseases Ghostwire: Tokyo confirms dog petting
Question: What's the latest news on space development?
Answer:

